In [13]:
import pandas as pd
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,[alerts@bank.com](mailto:alerts@bank.com),Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,[alerts@bank.com](mailto:alerts@bank.com),Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral
2,3,[no-reply@service.com](mailto:no-reply@service...,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,[sales@shop.com](mailto:sales@shop.com),Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,[no-reply@service.com](mailto:no-reply@service...,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


In [14]:
def email_assistant(email_text):
    text = email_text.lower()

    # Urgent / notify rules
    if any(word in text for word in ["urgent", "deadline", "submit", "overdue", "immediately", "alert"]):
        return "notify", "urgent"

    # Ignore rules
    elif any(word in text for word in ["thank you", "newsletter", "promotion", "sale", "congratulations"]):
        return "ignore", "neutral"

    # Default: normal emails
    else:
        return "respond", "polite"


In [15]:
# define Dangerous Actions

DANGEROUS_ACTIONS=["respond"]


In [16]:
#HITL CHECKPOINT LOGIC
def hitl_check(action):
    # HITL policy
    if action in ["respond", "delete"]:
        return "WAIT_FOR_HUMAN"
    return "AUTO_APPROVED"


In [17]:
# Simulate Human Approval
def human_decision():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"


In [18]:
# HITL checkpoint logic
results = []

for _, row in df.sample(10).iterrows():
    # Step 1: Generate AI action
    action, tone = email_assistant(row["body"])

    # Step 2: HITL check
    status = hitl_check(action)

    # Step 3: Human approval if required
    if status == "WAIT_FOR_HUMAN":
        approved = human_decision()
        final_action = action if approved else "blocked"
    else:
        final_action = action

    # Step 4: Store results
    results.append({
        "email_id": row["body"][:50],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status
    })
pd.DataFrame(results)

,email_id,ai_action,final_action,hitl_status
0,Reminder: The client meeting is scheduled at 1...,respond,respond,WAIT_FOR_HUMAN
1,"Hello team, please find the attached weekly re...",respond,respond,WAIT_FOR_HUMAN
2,"Hi, don't miss our sale with discounts up to 7...",ignore,ignore,AUTO_APPROVED
3,Security alert: multiple failed login attempts...,notify,notify,AUTO_APPROVED
4,Congratulations! You have been selected as a l...,ignore,ignore,AUTO_APPROVED
5,"Hi, don't miss our sale with discounts up to 7...",ignore,ignore,AUTO_APPROVED
6,Congratulations! You have been selected as a l...,ignore,ignore,AUTO_APPROVED
7,Your order #4920 has been shipped and is expec...,respond,respond,WAIT_FOR_HUMAN
8,Notice: Your account will be locked unless ver...,respond,respond,WAIT_FOR_HUMAN
9,Your order #2832 has been shipped and is expec...,respond,respond,WAIT_FOR_HUMAN


In [19]:
milestone3_hitl_output = pd.DataFrame(results)

# Save output
output_path = "../data/milestone3_hitl_output_arbind.csv"
milestone3_hitl_output.to_csv(output_path, index=False)

print("Milestone 3 HITL output saved at:", output_path)



Milestone 3 HITL output saved at: ../data/milestone3_hitl_output_arbind.csv
